# Wan 2.2 Генератор Длинных Видео (Ultimate Edition)

**Настройка:** ComfyUI + WanVideoWrapper на Google Colab T4 (16GB VRAM)

---

### Какой ноутбук выбрать?

> Это **максимальный** ноутбук, который включает ВСЕ возможности:
> генерация видео, пост-обработка, пакетная генерация, Google Drive, Telegram бот.
> Если вам нужен только базовый генератор без дополнительных инструментов,
> используйте `colab_wan_video.ipynb`.

---

### Пайплайн

```
Основной поток (обязательно):

  [1-2. GPU]  -->  [3-4. Установка]  -->  [5. Модели]  -->  [6. Воркфлоу]  -->  [7. Запуск]  -->  ComfyUI!
     |                  |                     |                  |                   |
   Проверка        ComfyUI +           Wan 2.2 14B         Скачивание         Cloudflare
   и профиль      кастомные ноды      + Text Encoder       воркфлоу            туннель

Опциональные инструменты (запускайте по необходимости):

  [Пост-обработка]    [Пакетная генерация]    [Google Drive]    [Telegram бот]
   Апскейл 4x           Несколько               Авто-            Генерация
   Интерполяция          промптов               сохранение        через чат
   fps                   подряд                 результатов       в Telegram
```

---

## Возможности
- **Генерация видео:** Режимы I2V, T2V, V2V через Wan 2.2 14B
- **Длинные видео:** До 30 секунд (721 кадров @ 24fps)
- **Авто-конфигурация GPU:** Определяет GPU и подбирает оптимальные параметры
- **Модульная загрузка:** Скачивайте только нужные модели
- **Пост-обработка:** Апскейл (480p->1080p), интерполяция кадров (24->48fps), экспорт через ffmpeg
- **Пакетная генерация:** Отправка нескольких промптов через ComfyUI API
- **Google Drive:** Автосохранение результатов
- **Монитор VRAM:** Отслеживание памяти в реальном времени
- **Telegram бот:** Отправь промпт/фото в TG -- получи видео автоматически

## Быстрый старт
1. Запустите ячейки 1-5 по порядку (GPU -> Установка -> Модели -> Воркфлоу -> Запуск)
2. Откройте URL туннеля Cloudflare
3. Загрузите воркфлоу и нажмите Queue
4. Используйте ячейки пост-обработки для апскейла/интерполяции
5. (Опционально) Запустите ячейки 11-12 для старта Telegram бота

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Нет'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ" if torch.cuda.is_available() else '')

In [ ]:
#@title 1.5 Авто-конфигурация GPU
import torch

GPU_PROFILES = {
    "T4":   {"blocks_to_swap": 20, "max_res": (832, 480),  "max_frames": 481, "dtype": "fp8",  "vram_gb": 16},
    "L4":   {"blocks_to_swap": 10, "max_res": (1024, 576), "max_frames": 601, "dtype": "fp8",  "vram_gb": 24},
    "A100": {"blocks_to_swap": 0,  "max_res": (1280, 720), "max_frames": 721, "dtype": "bf16", "vram_gb": 80},
    "V100": {"blocks_to_swap": 15, "max_res": (832, 480),  "max_frames": 481, "dtype": "fp16", "vram_gb": 16},
    "A10G": {"blocks_to_swap": 10, "max_res": (1024, 576), "max_frames": 601, "dtype": "fp8",  "vram_gb": 24},
}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0

# Подбор профиля GPU
GPU_CONFIG = GPU_PROFILES.get("T4")  # по умолчанию
for key, profile in GPU_PROFILES.items():
    if key.lower() in gpu_name.lower():
        GPU_CONFIG = profile
        break

# Переопределение при большем объёме VRAM (напр. Colab Pro иногда даёт больше)
if vram_gb > 20 and GPU_CONFIG["vram_gb"] <= 16:
    GPU_CONFIG["blocks_to_swap"] = max(GPU_CONFIG["blocks_to_swap"] - 5, 0)
    GPU_CONFIG["max_frames"] = 601

print(f"GPU: {gpu_name} ({vram_gb:.0f} ГБ)")
print(f"Профиль: blocks_to_swap={GPU_CONFIG['blocks_to_swap']}, "
      f"max_res={GPU_CONFIG['max_res']}, max_frames={GPU_CONFIG['max_frames']}, "
      f"dtype={GPU_CONFIG['dtype']}")
print(f"\nИспользуйте эти значения в нодах воркфлоу.")

In [ ]:
#@title 2. Установка ComfyUI
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

In [ ]:
#@title 3. Установка кастомных нод
%cd /content/ComfyUI/custom_nodes

# Основные ноды для видео
!test -d ComfyUI-WanVideoWrapper || git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

# Интерполяция кадров (RIFE)
!test -d ComfyUI-Frame-Interpolation || git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git

# Детекция лиц, сегментация, региональный апскейл
!test -d ComfyUI-Impact-Pack || git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git

# Установка зависимостей
import glob
for req in glob.glob("*/requirements.txt"):
    print(f"Устанавливаю {req}...")
    !pip install -r {req} -q -c /tmp/numpy_constraint.txt

# У Impact Pack отдельный скрипт установки
%cd /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack
!python install.py -q 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes

print("\nВсе кастомные ноды установлены!")

In [ ]:
#@title 3.5 Установка Sage Attention (~30% быстрее)
#@markdown Оптимизированное ядро внимания. Пропустите, если установка не удалась — всё работает и без него.
!pip install sageattention -q 2>/dev/null || echo "SageAttention недоступен для этого GPU, пропускаю (не критично)"
print("Установка Sage Attention завершена.")

In [ ]:
#@title 4. Скачивание моделей (модульное)
#@markdown ---
#@markdown ### Какие модели нужны?
#@markdown
#@markdown | Модель | Размер | Статус | Зачем нужна |
#@markdown |--------|--------|--------|-------------|
#@markdown | **Wan 2.2 T2V 14B fp8** | ~9.5 ГБ | ОБЯЗАТЕЛЬНО | Основная модель генерации видео |
#@markdown | **VACE модуль 14B** | ~5 ГБ | ОБЯЗАТЕЛЬНО | Модуль для длинных видео (>10 сек) |
#@markdown | **UMT5 XXL Text Encoder** | ~4.9 ГБ | ОБЯЗАТЕЛЬНО | Понимание текстовых промптов |
#@markdown | **Wan 2.2 VAE** | ~200 МБ | ОБЯЗАТЕЛЬНО | Декодирование латентов в видео |
#@markdown | RealESRGAN x4plus | ~64 МБ | Опционально | Апскейл 480p -> 1080p (пост-обработка) |
#@markdown | RIFE v4.6 | ~115 МБ | Опционально | Интерполяция кадров 24fps -> 48fps |
#@markdown
#@markdown ---
#@markdown **Итого обязательных:** ~19.6 ГБ (скачивание ~10-15 мин на Colab)
#@markdown
#@markdown Без первых 4 моделей генерация **не запустится**.
#@markdown Опциональные модели нужны только для пост-обработки (ячейка 7).

import os
import ipywidgets as widgets
from IPython.display import display

models_dir = "/content/ComfyUI/models"

# Реестр моделей: (метка чекбокса, размер, путь, url)
MODEL_REGISTRY = {
    "base_diffusion": {
        "label": "Wan 2.2 T2V 14B fp8 (ОБЯЗАТЕЛЬНО)",
        "size": "~9.5 ГБ",
        "path": f"{models_dir}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
        "default": True,
    },
    "vace_module": {
        "label": "VACE модуль 14B (длинное видео)",
        "size": "~5 ГБ",
        "path": f"{models_dir}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
        "default": True,
    },
    "text_encoder": {
        "label": "UMT5 XXL Text Encoder fp8 (ОБЯЗАТЕЛЬНО)",
        "size": "~4.9 ГБ",
        "path": f"{models_dir}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "default": True,
    },
    "vae": {
        "label": "Wan 2.2 VAE (ОБЯЗАТЕЛЬНО)",
        "size": "~200 МБ",
        "path": f"{models_dir}/vae/wan2.2_vae.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/wan2.2_vae.safetensors",
        "default": True,
    },
    "upscale_realesrgan": {
        "label": "RealESRGAN x4plus (апскейл видео)",
        "size": "~64 МБ",
        "path": f"{models_dir}/upscale_models/RealESRGAN_x4plus.pth",
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "default": True,
    },
    "rife_model": {
        "label": "RIFE v4.6 (интерполяция кадров)",
        "size": "~115 МБ",
        "path": f"{models_dir}/custom_nodes/ComfyUI-Frame-Interpolation/ckpts/rife/",
        "url": "AUTO",  # скачивается нодой автоматически при первом использовании
        "default": False,
    },
}

print("=" * 60)
print("  СКАЧИВАНИЕ МОДЕЛЕЙ")
print("=" * 60)
print()
print("  [ОБЯЗАТЕЛЬНО]  Wan 2.2 T2V 14B fp8     ~9.5 ГБ  -- основная модель")
print("  [ОБЯЗАТЕЛЬНО]  VACE модуль 14B          ~5.0 ГБ  -- длинные видео")
print("  [ОБЯЗАТЕЛЬНО]  UMT5 XXL Text Encoder    ~4.9 ГБ  -- текстовые промпты")
print("  [ОБЯЗАТЕЛЬНО]  Wan 2.2 VAE              ~0.2 ГБ  -- декодер видео")
print("  [опционально]  RealESRGAN x4plus        ~64 МБ   -- апскейл (пост-обработка)")
print("  [опционально]  RIFE v4.6                ~115 МБ  -- интерполяция fps")
print()
print("  Итого обязательных: ~19.6 ГБ")
print("  Примерное время скачивания: 10-15 мин")
print()
print("Выберите модели для скачивания:\n")

checkboxes = {}
for key, model in MODEL_REGISTRY.items():
    cb = widgets.Checkbox(
        value=model["default"],
        description=f'{model["label"]} ({model["size"]})',
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )
    checkboxes[key] = cb
    display(cb)

download_btn = widgets.Button(description="Скачать выбранные", button_style="success")
output = widgets.Output()
display(download_btn, output)

def on_download(btn):
    with output:
        output.clear_output()
        for key, cb in checkboxes.items():
            if not cb.value:
                continue
            model = MODEL_REGISTRY[key]
            if model["url"] == "AUTO":
                print(f"[пропуск] {model['label']} — скачивается автоматически при первом использовании")
                continue
            target_dir = os.path.dirname(model["path"])
            os.makedirs(target_dir, exist_ok=True)
            if os.path.exists(model["path"]):
                print(f"[есть] {model['label']}")
                continue
            print(f"[скачиваю] {model['label']}...")
            !wget -q --show-progress -O "{model['path']}" "{model['url']}"
            # Валидация скачивания
            if os.path.exists(model["path"]) and os.path.getsize(model["path"]) < 1024:
                os.remove(model["path"])
                print(f"  ОШИБКА: {model['label']} не скачан — перезапустите")
        print("\nГотово! Выбранные модели скачаны.")
        !du -sh {models_dir}/diffusion_models/* {models_dir}/text_encoders/* {models_dir}/vae/* {models_dir}/upscale_models/* 2>/dev/null || true

download_btn.on_click(on_download)

In [ ]:
#@title 4.5 Скачивание воркфлоу
import os
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
!wget -q -O /content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json \
  "https://gist.githubusercontent.com/russiankendricklamar/ea1ab4a53e1be1b8401e5031bcdae4f3/raw/video_wan_long_ultimate.json"
print("Воркфлоу скачан!")

In [ ]:
#@title 5. Запуск ComfyUI с туннелем Cloudflare
import subprocess, time, re, os, requests

# Борьба с фрагментацией CUDA памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI в фоне
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd='/content/ComfyUI',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
for i in range(12):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            break
    except Exception:
        pass
else:
    print("  ComfyUI не ответил за 60 сек — проверьте логи выше.")

# Запуск туннеля Cloudflare
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Поиск URL туннеля
for _ in range(10):
    line = tunnel_proc.stdout.readline().decode()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"\n{'='*60}")
        print(f"ComfyUI готов!")
        print(f"Откройте этот URL: {url}")
        print(f"{'='*60}")
        print(f"\nЗагрузите воркфлоу: video_wan_long_ultimate")
        break
else:
    print("URL туннеля пока не найден. Проверьте вывод 'cloudflared' вручную.")
    print("ComfyUI должен работать на порту 8188")

---

## Руководство по использованию

---

### Шаг 1. Откройте ComfyUI

1. Скопируйте URL из ячейки 5 (вида `https://xxx.trycloudflare.com`)
2. Откройте его в браузере

### Шаг 2. Загрузите воркфлоу

1. В ComfyUI нажмите **Menu** (верхний левый угол)
2. Нажмите **Load**
3. Выберите файл **`video_wan_long_ultimate`**

### Шаг 3. Настройте генерацию

1. Загрузите стартовое изображение в ноду **`LoadImage`** (для режима I2V)
2. Напишите промпт в ноде **`Text Conditioning`**
3. Установите количество кадров и разрешение (см. таблицы ниже)

### Шаг 4. Запустите!

Нажмите **Queue Prompt** и ждите результата.

---

### Таблица длительности видео

| Кадры | Длительность (24fps) | Время генерации (T4) | Рекомендация |
|:-----:|:--------------------:|:--------------------:|:------------:|
| **81** | 3.4 сек | ~2-3 мин | Тест / превью |
| **241** | 10 сек | ~5-8 мин | Короткий клип |
| **481** | 20 сек | ~10-15 мин | Стандарт |
| **721** | 30 сек | ~20-25 мин | Максимум (только A100/L4) |

> На бесплатном T4 рекомендуем **241-481 кадров**. 721 кадров требует >16 ГБ VRAM.

---

### Таблица разрешений

| Размер | Соотношение | Назначение | VRAM |
|:------:|:-----------:|:----------:|:----:|
| **832x480** | 16:9 | YouTube, горизонтальное | ~12 ГБ |
| **480x832** | 9:16 | Reels, TikTok, вертикальное | ~12 ГБ |
| **608x1080** | 9:16 | HD вертикальное | ~16 ГБ (нужен L4+) |

> После генерации можно апскейлить 480p -> 1080p через пост-обработку (ячейка 7).

---

### Дополнительные инструменты

| Ячейка | Инструмент | Что делает |
|:------:|:----------:|:-----------|
| **6** | Google Drive | Подключает Drive, `sync_to_drive()` копирует результаты |
| **7** | Пост-обработка | RealESRGAN 4x апскейл + RIFE/minterpolate интерполяция fps + экспорт MP4/WebM/GIF |
| **8** | Пакетная генерация | Один промпт на строку, отправляет в ComfyUI API по очереди |
| **9** | Монитор VRAM | Прогресс-бар VRAM, одиночная проверка или live-режим |
| **10** | Оптимизации | Кэш моделей на Drive, TeaCache 2-3x ускорение, шаблоны промптов |
| **11-12** | Telegram бот | Полная автоматизация: промпт/фото в TG -> видео в чат |

---

### Telegram бот (ячейки 11-12)

1. Получите токен бота у **@BotFather** в Telegram
2. Запустите ячейку 11 -- вставьте токен, укажите разрешённые ID пользователей
3. Запустите ячейку 12 -- бот стартует!
4. Откройте бота в Telegram, нажмите **Start**
5. Выберите **"Написать -> Видео"** или **"Картинка -> Видео"**
6. Выберите длительность и формат
7. Ждите -- бот отправит готовое видео прямо в чат
8. Видео также автоматически сохраняется на Google Drive

In [ ]:
#@title 6. Подключение Google Drive (авто-сохранение)
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

DRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

COMFY_OUTPUT = "/content/ComfyUI/output"

def sync_to_drive():
    """Копирует все новые файлы из вывода ComfyUI на Google Drive."""
    if not os.path.exists(COMFY_OUTPUT):
        print("Папка вывода ещё не создана. Сначала сгенерируйте видео.")
        return
    files = os.listdir(COMFY_OUTPUT)
    if not files:
        print("Нет файлов для синхронизации.")
        return
    copied = 0
    for f in files:
        src = os.path.join(COMFY_OUTPUT, f)
        dst = os.path.join(DRIVE_OUTPUT, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
            print(f"  Скопировано: {f}")
    print(f"\nСинхронизировано {copied} новых файл(ов) в {DRIVE_OUTPUT}")

print(f"Drive подключён. Результаты будут сохраняться в: {DRIVE_OUTPUT}")
print("Вызовите sync_to_drive() в любой момент для копирования новых результатов на Drive.")

In [ ]:
#@title 7. Пост-обработка (Апскейл + Интерполяция + Экспорт)
import os
import subprocess

COMFY_OUTPUT = "/content/ComfyUI/output"
PROCESSED_DIR = "/content/ComfyUI/output_processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

#@markdown ### Настройки
input_video = "" #@param {type:"string"}
enable_upscale = True #@param {type:"boolean"}
upscale_factor = 4 #@param [2, 4] {type:"raw"}
enable_interpolation = False #@param {type:"boolean"}
target_fps = 48 #@param [48, 60] {type:"raw"}
add_audio = "" #@param {type:"string"}
output_format = "mp4" #@param ["mp4", "webm", "gif"]

def parse_fps(probe_output):
    """Безопасно парсит fps из вывода ffprobe вида '24/1' или '30000/1001'."""
    try:
        text = probe_output.strip()
        if "/" in text:
            num, den = text.split("/")
            return int(num) / int(den)
        return float(text)
    except (ValueError, ZeroDivisionError):
        return 24

# Автоопределение последнего видео, если путь не указан
if not input_video:
    videos = sorted(
        [f for f in os.listdir(COMFY_OUTPUT) if f.endswith((".mp4", ".webm", ".png"))],
        key=lambda f: os.path.getmtime(os.path.join(COMFY_OUTPUT, f)),
        reverse=True,
    )
    if videos:
        input_video = os.path.join(COMFY_OUTPUT, videos[0])
        print(f"Автоопределено: {input_video}")
    else:
        print("Видео не найдены в папке вывода. Сначала сгенерируйте видео.")

if input_video and os.path.exists(input_video):
    basename = os.path.splitext(os.path.basename(input_video))[0]
    current = input_video

    # Шаг 1: Апскейл через RealESRGAN
    if enable_upscale:
        print("\n[1/3] Апскейл через RealESRGAN...")
        !pip install realesrgan -q 2>/dev/null
        upscaled = os.path.join(PROCESSED_DIR, f"{basename}_upscaled.mp4")

        # Извлечение кадров, апскейл, сборка
        frames_dir = os.path.join(PROCESSED_DIR, "frames_in")
        frames_up_dir = os.path.join(PROCESSED_DIR, "frames_up")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(frames_up_dir, exist_ok=True)

        !ffmpeg -y -i "{current}" -qscale:v 2 "{frames_dir}/%06d.png" -loglevel error
        !python -m realesrgan -i "{frames_dir}" -o "{frames_up_dir}" -n RealESRGAN_x4plus -s {upscale_factor} --suffix "" 2>/dev/null || \
            !realesrgan-ncnn-vulkan -i "{frames_dir}" -o "{frames_up_dir}" -n realesrgan-x4plus -s {upscale_factor} 2>/dev/null

        # Безопасное получение оригинального fps
        fps_str = !ffprobe -v error -select_streams v -of csv=p=0 -show_entries stream=r_frame_rate "{current}"
        fps = parse_fps(fps_str[0]) if fps_str else 24

        !ffmpeg -y -framerate {fps} -i "{frames_up_dir}/%06d.png" -c:v libx264 -pix_fmt yuv420p "{upscaled}" -loglevel error
        current = upscaled
        print(f"  Апскейл завершён: {upscaled}")

        # Очистка временных кадров
        !rm -rf "{frames_dir}" "{frames_up_dir}"

    # Шаг 2: Интерполяция кадров через RIFE
    if enable_interpolation:
        print("\n[2/3] Интерполяция кадров через RIFE...")
        interpolated = os.path.join(PROCESSED_DIR, f"{basename}_interpolated.mp4")
        !pip install rife-ncnn-vulkan-python -q 2>/dev/null || true

        # Используем ffmpeg + RIFE через minterpolate как запасной вариант
        !ffmpeg -y -i "{current}" -vf "minterpolate=fps={target_fps}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1" \
            -c:v libx264 -pix_fmt yuv420p "{interpolated}" -loglevel error
        current = interpolated
        print(f"  Интерполяция до {target_fps}fps: {interpolated}")

    # Шаг 3: Финальный экспорт через ffmpeg
    print("\n[3/3] Финальный экспорт...")
    final = os.path.join(PROCESSED_DIR, f"{basename}_final.{output_format}")

    audio_flag = f'-i "{add_audio}" -c:a aac -shortest' if add_audio and os.path.exists(add_audio) else "-an"

    if output_format == "gif":
        !ffmpeg -y -i "{current}" -vf "fps=15,scale=480:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse" \
            "{final}" -loglevel error
    elif output_format == "webm":
        !ffmpeg -y -i "{current}" {audio_flag} -c:v libvpx-vp9 -crf 30 -b:v 0 "{final}" -loglevel error
    else:
        !ffmpeg -y -i "{current}" {audio_flag} -c:v libx264 -crf 18 -preset slow -pix_fmt yuv420p "{final}" -loglevel error

    print(f"\nГотово! Финальное видео: {final}")
    file_size = os.path.getsize(final) / (1024 * 1024)
    print(f"Размер файла: {file_size:.1f} МБ")

    # Автосинхронизация на Drive
    if os.path.exists("/content/drive/MyDrive/ComfyUI_Output"):
        import shutil
        shutil.copy2(final, f"/content/drive/MyDrive/ComfyUI_Output/{os.path.basename(final)}")
        print(f"Сохранено на Google Drive!")

In [ ]:
#@title 8. Пакетная генерация (ComfyUI API)
import json
import urllib.request
import urllib.parse
import time

COMFYUI_URL = "http://localhost:8188"

#@markdown ### Промпты (по одному на строку)
prompts_text = """a beautiful woman walking through a cherry blossom garden, cinematic, slow motion
a futuristic city skyline at sunset, neon lights, flying cars, cyberpunk style
ocean waves crashing on rocks, golden hour, dramatic slow motion""" #@param {type:"string"}

#@markdown ### Настройки
num_frames = 241 #@param {type:"integer"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
steps = 30 #@param {type:"integer"}
cfg = 6.0 #@param {type:"number"}

def get_workflow():
    """Загрузить JSON-шаблон воркфлоу."""
    workflow_path = "/content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json"
    with open(workflow_path, "r") as f:
        return json.load(f)

def queue_prompt(workflow_json):
    """Отправить воркфлоу в очередь ComfyUI."""
    data = json.dumps({"prompt": workflow_json}).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFYUI_URL}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = urllib.request.urlopen(req)
    return json.loads(resp.read())

def get_queue_status():
    """Проверить текущий статус очереди."""
    resp = urllib.request.urlopen(f"{COMFYUI_URL}/queue")
    data = json.loads(resp.read())
    running = len(data.get("queue_running", []))
    pending = len(data.get("queue_pending", []))
    return running, pending

def wait_for_completion(prompt_id, timeout=3600):
    """Ожидание завершения конкретного промпта."""
    start = time.time()
    while time.time() - start < timeout:
        resp = urllib.request.urlopen(f"{COMFYUI_URL}/history/{prompt_id}")
        history = json.loads(resp.read())
        if prompt_id in history:
            return history[prompt_id]
        time.sleep(5)
    return None

prompts = [p.strip() for p in prompts_text.strip().split("\n") if p.strip()]
print(f"Отправляю {len(prompts)} промптов в очередь...\n")

for i, prompt in enumerate(prompts):
    print(f"[{i+1}/{len(prompts)}] {prompt[:80]}...")

    try:
        workflow = get_workflow()

        # Обновление текста промпта в нодах воркфлоу
        # (ID нод могут отличаться — ищем текстовые поля)
        for node_id, node in workflow.items():
            if isinstance(node, dict):
                inputs = node.get("inputs", {})
                # Ищем поля text/prompt
                for key in ["text", "prompt", "positive"]:
                    if key in inputs and isinstance(inputs[key], str):
                        inputs[key] = prompt

        result = queue_prompt(workflow)
        prompt_id = result.get("prompt_id", "unknown")
        print(f"  В очереди: {prompt_id}")

        # Ожидание завершения
        print(f"  Генерация...", end="", flush=True)
        history = wait_for_completion(prompt_id)
        if history:
            print(f" Готово!")
        else:
            print(f" Таймаут (всё ещё выполняется в фоне)")

    except Exception as e:
        print(f"  Ошибка: {e}")

print(f"\nПакетная генерация завершена! Проверьте /content/ComfyUI/output/ для результатов.")
print("Вызовите sync_to_drive() для копирования результатов на Google Drive.")

In [ ]:
#@title 9. Монитор VRAM
import torch
import time
from IPython.display import clear_output

def vram_status():
    """Показать текущее использование VRAM."""
    if not torch.cuda.is_available():
        print("CUDA GPU недоступна")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = total - reserved
    pct = (reserved / total) * 100

    bar_len = 40
    filled = int(bar_len * reserved / total)
    bar = "=" * filled + "-" * (bar_len - filled)

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: [{bar}] {pct:.0f}%")
    print(f"  Выделено:      {allocated:.1f} ГБ")
    print(f"  Зарезервировано: {reserved:.1f} ГБ")
    print(f"  Свободно:      {free:.1f} ГБ / {total:.1f} ГБ всего")

#@markdown ### Режим
live_monitor = False #@param {type:"boolean"}
refresh_sec = 5 #@param {type:"integer"}

if live_monitor:
    try:
        while True:
            clear_output(wait=True)
            vram_status()
            time.sleep(refresh_sec)
    except KeyboardInterrupt:
        print("Мониторинг остановлен.")
else:
    vram_status()

In [ ]:
#@title 10. Набор оптимизаций (Скорость + VRAM + Качество)
import os, json, shutil, torch, gc

COMFY_OUTPUT = "/content/ComfyUI/output"
DRIVE_CACHE = "/content/drive/MyDrive/ComfyUI_ModelCache"
models_dir = "/content/ComfyUI/models"

# ========== 1. КЭШ МОДЕЛЕЙ НА DRIVE ==========
#@markdown ### Кэширование моделей на Google Drive (пропуск скачивания 20ГБ в следующей сессии)
enable_model_cache = True #@param {type:"boolean"}

def cache_models_to_drive():
    """Копирует скачанные модели на Drive для повторного использования."""
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    for subdir in ["diffusion_models", "text_encoders", "vae", "upscale_models"]:
        src = os.path.join(models_dir, subdir)
        dst = os.path.join(DRIVE_CACHE, subdir)
        if not os.path.exists(src):
            continue
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            src_f, dst_f = os.path.join(src, f), os.path.join(dst, f)
            if os.path.isfile(src_f) and not os.path.exists(dst_f):
                print(f"  Кэширую {f}...")
                shutil.copy2(src_f, dst_f)
    print("Модели закэшированы на Drive!")

def restore_models_from_drive():
    """Восстанавливает модели из кэша Drive (мгновенно, без скачивания)."""
    if not os.path.exists(DRIVE_CACHE):
        print("Кэш на Drive не найден.")
        return False
    restored = 0
    for subdir in ["diffusion_models", "text_encoders", "vae", "upscale_models"]:
        src = os.path.join(DRIVE_CACHE, subdir)
        dst = os.path.join(models_dir, subdir)
        if not os.path.exists(src):
            continue
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            src_f, dst_f = os.path.join(src, f), os.path.join(dst, f)
            if os.path.isfile(src_f) and not os.path.exists(dst_f):
                print(f"  Восстанавливаю {f}...")
                os.symlink(src_f, dst_f)
                restored += 1
    if restored:
        print(f"Восстановлено {restored} моделей из кэша Drive (симлинки)!")
    else:
        print("Все модели уже на месте.")
    return restored > 0

if enable_model_cache and os.path.exists("/content/drive"):
    restore_models_from_drive()

# ========== 2. НАСТРОЙКА TEACACHE ==========
#@markdown ### TeaCache (генерация в 2-3 раза быстрее)
#@markdown Включите в ноде WanVideoWrapper: установите `enable_teacache = true`
teacache_threshold = 0.3 #@param {type:"number"}
print(f"\nTeaCache: Установите threshold={teacache_threshold} в ноде WanVideoWrapper")
print("  Ниже = быстрее, но хуже качество. Диапазон: 0.1 (быстро) - 0.5 (качество)")

# ========== 3. ОПТИМАЛЬНЫЕ НАСТРОЙКИ ГЕНЕРАЦИИ ==========
#@markdown ### Рекомендуемые настройки воркфлоу для T4
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 16

print(f"\n{'='*50}")
print(f"ОПТИМАЛЬНЫЕ НАСТРОЙКИ ДЛЯ ВАШЕГО GPU ({vram_gb:.0f}ГБ VRAM)")
print(f"{'='*50}")

if vram_gb <= 16:
    print("""
Нода WanVideoWrapper:
  blocks_to_swap: 20 (увеличьте до 30 при OOM)
  enable_teacache: true
  teacache_threshold: 0.3
  attention_mode: sage (если установлен) или sdpa

Нода VACEEncode:
  width: 832, height: 480
  num_frames: 241 (10 сек) или 481 (20 сек)

Сэмплер:
  steps: 20 (не 30 — планировщик Flow Match сходится быстро)
  cfg: 5.0-6.0
  scheduler: flow_match
  
VAE Decode:
  enable_tiling: true (экономит ~2ГБ VRAM)
  tile_size: 256
""")
elif vram_gb <= 24:
    print("""
Нода WanVideoWrapper:
  blocks_to_swap: 10
  enable_teacache: true
  teacache_threshold: 0.3
  attention_mode: sage (если установлен) или sdpa

VACEEncode: 1024x576, до 601 кадров
Сэмплер: steps=25, cfg=5.5, scheduler=flow_match
VAE Decode: enable_tiling=true, tile_size=512
""")
else:
    print("""
Нода WanVideoWrapper:
  blocks_to_swap: 0
  enable_teacache: true
  teacache_threshold: 0.2
  attention_mode: sage

VACEEncode: 1280x720, до 721 кадров  
Сэмплер: steps=30, cfg=5.0, scheduler=flow_match
VAE Decode: enable_tiling=false
""")

# ========== 4. ШАБЛОНЫ НЕГАТИВНЫХ ПРОМПТОВ ==========
print(f"\n{'='*50}")
print("ШАБЛОНЫ НЕГАТИВНЫХ ПРОМПТОВ")
print(f"{'='*50}")
print("""
Общий (скопируйте в негативный промпт):
  "blurry, low quality, distorted, deformed, ugly, bad anatomy, watermark, text, logo, oversaturated"

Реалистичный стиль:
  "cartoon, anime, illustration, painting, drawing, 3d render, cgi, unrealistic, plastic skin"

Аниме стиль:
  "photo, realistic, photograph, 3d, ugly, deformed, noisy, blurry, low contrast"

Портрет/Лицо:
  "deformed face, extra fingers, mutated hands, bad proportions, extra limbs, cross-eyed, blurry face"
""")

# ========== 5. CFG RESCALE ==========
print(f"{'='*50}")
print("CFG RESCALE (борьба с перенасыщением)")
print(f"{'='*50}")
print("""
Если цвета перенасыщены при высоком CFG (>6):
  Добавьте ноду 'RescaleCFG' между сэмплером и моделью
  Установите rescale_multiplier: 0.7
  Это сохраняет детали, уменьшая засветку цветов
""")

# ========== 6. АВТОВОССТАНОВЛЕНИЕ ПРИ OOM ==========
def clear_vram():
    """Экстренная очистка VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("VRAM очищена. При повторном OOM попробуйте более низкие настройки.")
    print("  Попробуйте: blocks_to_swap=30, resolution=480x320, num_frames=81")

print(f"{'='*50}")
print("ВОССТАНОВЛЕНИЕ ПРИ OOM: Вызовите clear_vram() при ошибке нехватки памяти")
print(f"{'='*50}")

if enable_model_cache and os.path.exists("/content/drive"):
    print("\nСовет: Вызовите cache_models_to_drive() после скачивания моделей")
    print("     В следующей сессии модели восстановятся мгновенно через симлинки!")

In [ ]:
#@title 11. Настройка Telegram бота
#@markdown ---
#@markdown ### Как получить токен бота (пошагово)
#@markdown
#@markdown 1. Откройте Telegram и найдите **@BotFather**
#@markdown 2. Нажмите **Start** (или отправьте `/start`)
#@markdown 3. Отправьте команду `/newbot`
#@markdown 4. Введите **имя бота** (отображаемое имя, например: `Мой Видео Бот`)
#@markdown 5. Введите **username бота** (должен заканчиваться на `bot`, например: `my_video_gen_bot`)
#@markdown 6. BotFather пришлёт **токен** вида: `7123456789:AAF...xyz`
#@markdown 7. Скопируйте этот токен и вставьте в поле ниже
#@markdown
#@markdown ---
#@markdown ### Как узнать свой Telegram ID
#@markdown
#@markdown 1. Найдите в Telegram бота **@userinfobot**
#@markdown 2. Нажмите **Start**
#@markdown 3. Бот ответит вашим числовым ID (например: `123456789`)
#@markdown 4. Впишите его в поле `ALLOWED_USERS` ниже
#@markdown
#@markdown > Без указания ID бот будет доступен **всем** -- это небезопасно!
#@markdown
#@markdown ---

import getpass

print("=" * 60)
print("  НАСТРОЙКА TELEGRAM БОТА")
print("=" * 60)
print()
print("  Шаг 1: Получите токен у @BotFather (см. инструкцию выше)")
print("  Шаг 2: Вставьте токен в поле ниже")
print("  Шаг 3: Укажите Telegram ID разрешённых пользователей")
print("  Шаг 4: Запустите следующую ячейку (12) для старта бота")
print()

BOT_TOKEN = getpass.getpass("Вставьте токен Telegram бота: ")

#@markdown ### Кто может использовать бота?
#@markdown Введите Telegram ID пользователей через запятую.
#@markdown Чтобы узнать свой ID: напишите @userinfobot в Telegram.
ALLOWED_USERS = "123456789, 987654321" #@param {type:"string"}
ALLOWED_USER_IDS = [int(uid.strip()) for uid in ALLOWED_USERS.split(",") if uid.strip()]

#@markdown ### Настройки по умолчанию
MAX_QUEUE_SIZE = 10 #@param {type:"integer"}
DEFAULT_STEPS = 20 #@param {type:"integer"}
DEFAULT_CFG = 5.5 #@param {type:"number"}
ENABLE_UPSCALE = True #@param {type:"boolean"}
ENABLE_INTERPOLATION = True #@param {type:"boolean"}
TARGET_FPS = 48 #@param {type:"integer"}

COMFYUI_URL = "http://localhost:8188"
COMFY_OUTPUT = "/content/ComfyUI/output"
PROCESSED_DIR = "/content/ComfyUI/output_processed"
DRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_Output"
WORKFLOW_PATH = "/content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json"

# Маппинг длительности/разрешения (скрыт от пользователя)
DURATION_MAP = {
    "short": {"frames": 81, "label": "Короткое (3 сек)", "est_min": 3},
    "medium": {"frames": 241, "label": "Среднее (10 сек)", "est_min": 5},
    "long": {"frames": 481, "label": "Длинное (20 сек)", "est_min": 10},
}

ORIENTATION_MAP = {
    "horizontal": {"width": 832, "height": 480, "label": "Горизонтальное"},
    "vertical": {"width": 480, "height": 832, "label": "Вертикальное"},
}

print()
print("-" * 60)
token_status = "OK" if BOT_TOKEN else "НЕ УКАЗАН"
print(f"  Токен бота:            {token_status}")
print(f"  Разрешённые ID:        {ALLOWED_USER_IDS}")
print(f"  Макс. очередь:         {MAX_QUEUE_SIZE}")
print(f"  Шаги / CFG:            {DEFAULT_STEPS} / {DEFAULT_CFG}")
print(f"  Апскейл:               {'Да' if ENABLE_UPSCALE else 'Нет'}")
print(f"  Интерполяция:          {'Да' if ENABLE_INTERPOLATION else 'Нет'} ({TARGET_FPS} fps)")
print("-" * 60)
print()
if BOT_TOKEN:
    print("  Всё готово! Запустите следующую ячейку (12) для старта бота.")
else:
    print("  ВНИМАНИЕ: токен не указан. Бот не запустится без токена.")

In [ ]:
#@title 12. Запуск Telegram бота
#@markdown Запустите эту ячейку для старта бота. Он будет работать до остановки.
#@markdown
#@markdown **Как использовать:** Откройте бота в Telegram и нажмите Start!

import os
import json
import time
import copy
import asyncio
import shutil
import subprocess
import urllib.request
import logging
from collections import deque

# Установка python-telegram-bot
!pip install python-telegram-bot==21.6 -q

from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    Application,
    CommandHandler,
    CallbackQueryHandler,
    MessageHandler,
    ConversationHandler,
    ContextTypes,
    filters,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("VideoBot")

# ==================== ИНТРО ВИДЕО ====================
INTRO_VIDEO_URL = "https://youtu.be/SxnjiI-3GsE"

# ==================== СОСТОЯНИЯ ====================
CHOOSE_MODE, WAIT_PHOTO, WAIT_PROMPT, CHOOSE_DURATION, CHOOSE_ORIENTATION = range(5)

# ==================== ОЧЕРЕДЬ ЗАДАЧ ====================
job_queue = asyncio.Queue()
generation_times = deque(maxlen=20)

def estimate_wait(position):
    """Оценка времени ожидания на основе прошлых генераций."""
    if not generation_times:
        return position * 5
    avg = sum(generation_times) / len(generation_times)
    return int(position * avg)

# ==================== КОНТРОЛЬ ДОСТУПА ====================
def authorized(func):
    """Декоратор: допускает только пользователей из белого списка."""
    async def wrapper(update: Update, context: ContextTypes.DEFAULT_TYPE):
        user_id = update.effective_user.id
        if ALLOWED_USER_IDS and user_id not in ALLOWED_USER_IDS:
            text = "У тебя нет доступа к этому боту."
            if update.callback_query:
                await update.callback_query.answer(text, show_alert=True)
            else:
                await update.message.reply_text(text)
            return ConversationHandler.END
        return await func(update, context)
    return wrapper

# ==================== ОБРАБОТЧИКИ ====================

@authorized
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Точка входа — отправка интро-видео и показ главного меню."""
    context.user_data.clear()

    # Отправка интро-видео (сцена активации Джарвиса)
    await update.message.reply_video(
        video=INTRO_VIDEO_URL,
        caption="Система активирована."
    )

    keyboard = [
        [InlineKeyboardButton("Написать -> Видео", callback_data="mode_t2v")],
        [InlineKeyboardButton("Картинка -> Видео", callback_data="mode_i2v")],
    ]
    await update.message.reply_text(
        "Привет! Я делаю видео из текста и картинок.\n\n"
        "Что хочешь сделать?\n"
        "(Отправь /cancel чтобы отменить в любой момент)",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_MODE

@authorized
async def restart_entry(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Обработка 'Сделать ещё' / 'Попробовать снова' — повторный вход в диалог."""
    query = update.callback_query
    await query.answer()
    context.user_data.clear()
    keyboard = [
        [InlineKeyboardButton("Написать -> Видео", callback_data="mode_t2v")],
        [InlineKeyboardButton("Картинка -> Видео", callback_data="mode_i2v")],
    ]
    await query.edit_message_text(
        "Что хочешь сделать?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_MODE

@authorized
async def choose_mode(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Пользователь выбрал T2V или I2V."""
    query = update.callback_query
    await query.answer()
    mode = query.data
    context.user_data["mode"] = mode

    if mode == "mode_i2v":
        await query.edit_message_text(
            "Отправь картинку, которую хочешь оживить."
        )
        return WAIT_PHOTO
    else:
        await query.edit_message_text(
            "Напиши, что должно быть на видео.\n\n"
            "Например: кот летит в космосе на ракете"
        )
        return WAIT_PROMPT

@authorized
async def receive_photo(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Пользователь отправил фото для I2V."""
    photo = update.message.photo[-1]
    file = await photo.get_file()
    photo_path = f"/content/ComfyUI/input/tg_{update.effective_user.id}_{int(time.time())}.jpg"
    os.makedirs("/content/ComfyUI/input", exist_ok=True)
    await file.download_to_drive(photo_path)

    if not os.path.exists(photo_path):
        await update.message.reply_text("Не удалось скачать фото. Попробуй ещё раз.")
        return WAIT_PHOTO

    context.user_data["image_path"] = photo_path

    await update.message.reply_text(
        "Отлично! Теперь напиши, что должно происходить на видео.\n\n"
        "Например: девушка поворачивает голову и улыбается"
    )
    return WAIT_PROMPT

@authorized
async def receive_prompt(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Пользователь отправил текстовый промпт."""
    context.user_data["prompt"] = update.message.text

    keyboard = [
        [InlineKeyboardButton("Короткое (3 сек)", callback_data="dur_short")],
        [InlineKeyboardButton("Среднее (10 сек)", callback_data="dur_medium")],
        [InlineKeyboardButton("Длинное (20 сек)", callback_data="dur_long")],
    ]
    await update.message.reply_text(
        "Как долго видео?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_DURATION

@authorized
async def choose_duration(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Пользователь выбрал длительность."""
    query = update.callback_query
    await query.answer()
    dur_key = query.data.replace("dur_", "")
    context.user_data["duration"] = dur_key

    keyboard = [
        [InlineKeyboardButton("Горизонтальное (YouTube)", callback_data="orient_horizontal")],
        [InlineKeyboardButton("Вертикальное (Reels/TikTok)", callback_data="orient_vertical")],
    ]
    await query.edit_message_text(
        "Какой формат?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_ORIENTATION

@authorized
async def choose_orientation(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Пользователь выбрал ориентацию — отправка задачи в очередь."""
    query = update.callback_query
    await query.answer()
    orient_key = query.data.replace("orient_", "")
    context.user_data["orientation"] = orient_key

    if job_queue.qsize() >= MAX_QUEUE_SIZE:
        await query.edit_message_text(
            "Очередь полная, попробуй чуть позже.\n\n"
            "Нажми /start чтобы попробовать снова."
        )
        return ConversationHandler.END

    dur = DURATION_MAP[context.user_data["duration"]]
    orient = ORIENTATION_MAP[orient_key]
    job = {
        "user_id": update.effective_user.id,
        "chat_id": update.effective_chat.id,
        "mode": context.user_data["mode"],
        "prompt": context.user_data["prompt"],
        "image_path": context.user_data.get("image_path"),
        "frames": dur["frames"],
        "width": orient["width"],
        "height": orient["height"],
        "submitted_at": time.time(),
    }

    position = job_queue.qsize() + 1
    est = estimate_wait(position)

    await job_queue.put(job)

    if position == 1:
        await query.edit_message_text("Делаю видео... Пришлю сюда когда будет готово!")
    else:
        await query.edit_message_text(
            f"Принято! Ты {position}-й в очереди.\n"
            f"Примерное ожидание: ~{est} мин.\n\n"
            f"Я напишу когда начну делать твоё!"
        )

    return ConversationHandler.END

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Отмена текущего диалога."""
    context.user_data.clear()
    await update.message.reply_text(
        "Отменено. Нажми /start чтобы начать заново."
    )
    return ConversationHandler.END

@authorized
async def queue_status(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Показать текущий статус очереди."""
    size = job_queue.qsize()
    if size == 0:
        await update.message.reply_text("Очередь пустая. Отправь /start чтобы сделать видео!")
    else:
        await update.message.reply_text(f"В очереди: {size} видео.")

# ==================== COMFYUI API ====================

def load_workflow():
    """Загрузить JSON-шаблон воркфлоу."""
    with open(WORKFLOW_PATH, "r") as f:
        return json.load(f)

def send_to_comfyui(workflow_json):
    """Отправить воркфлоу в очередь ComfyUI, вернуть prompt_id."""
    data = json.dumps({"prompt": workflow_json}).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFYUI_URL}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = urllib.request.urlopen(req, timeout=30)
    return json.loads(resp.read()).get("prompt_id")

def poll_completion(prompt_id, timeout=3600):
    """Опрос /history ComfyUI до завершения промпта."""
    start_t = time.time()
    while time.time() - start_t < timeout:
        try:
            resp = urllib.request.urlopen(
                f"{COMFYUI_URL}/history/{prompt_id}", timeout=15
            )
            history = json.loads(resp.read())
            if prompt_id in history:
                return history[prompt_id]
        except Exception:
            pass
        time.sleep(5)
    return None

def find_output_from_history(history):
    """Извлечь путь выходного файла из ответа истории ComfyUI."""
    for node_id, node_data in history.get("outputs", {}).items():
        for output_type in ("gifs", "images", "videos"):
            items = node_data.get(output_type, [])
            for item in items:
                filename = item.get("filename")
                if filename:
                    subfolder = item.get("subfolder", "")
                    path = os.path.join(COMFY_OUTPUT, subfolder, filename)
                    if os.path.exists(path):
                        return path
    return None

def find_latest_output():
    """Запасной вариант: найти последний созданный файл в папке вывода ComfyUI."""
    if not os.path.exists(COMFY_OUTPUT):
        return None
    files = [
        os.path.join(COMFY_OUTPUT, f)
        for f in os.listdir(COMFY_OUTPUT)
        if f.endswith((".mp4", ".webm", ".png", ".gif"))
    ]
    if not files:
        return None
    return max(files, key=os.path.getmtime)

# ==================== ПОСТ-ОБРАБОТКА ====================

def parse_fps(probe_output):
    """Безопасно парсит fps из вывода ffprobe вида '24/1' или '30000/1001'."""
    try:
        text = probe_output.strip()
        if "/" in text:
            num, den = text.split("/")
            return int(num) / int(den)
        return float(text)
    except (ValueError, ZeroDivisionError):
        return 24

def postprocess_video(input_path):
    """Апскейл + интерполяция + экспорт. Возвращает путь к финальному mp4."""
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    basename = os.path.splitext(os.path.basename(input_path))[0]
    current = input_path

    if ENABLE_UPSCALE:
        logger.info("Апскейл...")
        frames_dir = os.path.join(PROCESSED_DIR, "frames_in")
        frames_up_dir = os.path.join(PROCESSED_DIR, "frames_up")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(frames_up_dir, exist_ok=True)

        subprocess.run(
            ["ffmpeg", "-y", "-i", current, "-qscale:v", "2",
             f"{frames_dir}/%06d.png", "-loglevel", "error"],
            check=True
        )

        result = subprocess.run(
            ["python", "-m", "realesrgan", "-i", frames_dir, "-o", frames_up_dir,
             "-n", "RealESRGAN_x4plus", "-s", "4", "--suffix", ""],
            capture_output=True
        )
        if result.returncode != 0:
            logger.warning("RealESRGAN не сработал, пропускаю апскейл")
            frames_up_dir = frames_dir

        probe = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v",
             "-of", "csv=p=0", "-show_entries", "stream=r_frame_rate", current],
            capture_output=True, text=True
        )
        fps = parse_fps(probe.stdout)

        upscaled = os.path.join(PROCESSED_DIR, f"{basename}_up.mp4")
        subprocess.run(
            ["ffmpeg", "-y", "-framerate", str(fps),
             "-i", f"{frames_up_dir}/%06d.png",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", upscaled,
             "-loglevel", "error"],
            check=True
        )
        current = upscaled

        shutil.rmtree(frames_dir, ignore_errors=True)
        if frames_up_dir != frames_dir:
            shutil.rmtree(frames_up_dir, ignore_errors=True)

    if ENABLE_INTERPOLATION:
        logger.info("Интерполяция...")
        interpolated = os.path.join(PROCESSED_DIR, f"{basename}_interp.mp4")
        subprocess.run(
            ["ffmpeg", "-y", "-i", current,
             "-vf", f"minterpolate=fps={TARGET_FPS}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", interpolated,
             "-loglevel", "error"],
            check=True
        )
        current = interpolated

    final = os.path.join(PROCESSED_DIR, f"{basename}_final.mp4")
    subprocess.run(
        ["ffmpeg", "-y", "-i", current, "-an",
         "-c:v", "libx264", "-crf", "18", "-preset", "slow",
         "-pix_fmt", "yuv420p", final,
         "-loglevel", "error"],
        check=True
    )

    return final

def sync_file_to_drive(file_path):
    """Копирует один файл на Google Drive."""
    if os.path.exists(DRIVE_OUTPUT):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(file_path))
        shutil.copy2(file_path, dst)
        return True
    return False

# ==================== ВОРКЕР ====================

async def worker(app):
    """Обрабатывает задачи из очереди по одной."""
    logger.info("Воркер запущен, ожидание задач...")
    while True:
        job = await job_queue.get()
        start_time = time.time()

        try:
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Начинаю делать твоё видео..."
            )

            # 1. Сборка воркфлоу (deepcopy для избежания мутаций)
            workflow = copy.deepcopy(load_workflow())

            for node_id, node in workflow.items():
                if not isinstance(node, dict):
                    continue
                inputs = node.get("inputs", {})
                for key in ["text", "prompt", "positive"]:
                    if key in inputs and isinstance(inputs[key], str):
                        inputs[key] = job["prompt"]

            for node_id, node in workflow.items():
                if not isinstance(node, dict):
                    continue
                inputs = node.get("inputs", {})
                if "width" in inputs and "height" in inputs:
                    inputs["width"] = job["width"]
                    inputs["height"] = job["height"]
                if "num_frames" in inputs:
                    inputs["num_frames"] = job["frames"]
                if "steps" in inputs and isinstance(inputs["steps"], (int, float)):
                    inputs["steps"] = DEFAULT_STEPS
                if "cfg" in inputs and isinstance(inputs["cfg"], (int, float)):
                    inputs["cfg"] = DEFAULT_CFG

            if job["mode"] == "mode_i2v" and job.get("image_path"):
                for node_id, node in workflow.items():
                    if not isinstance(node, dict):
                        continue
                    if node.get("class_type") in ("LoadImage", "Load Image"):
                        node["inputs"]["image"] = job["image_path"]

            # 2. Отправка в ComfyUI
            prompt_id = send_to_comfyui(workflow)
            if not prompt_id:
                raise RuntimeError("ComfyUI не вернул prompt_id")

            # 3. Ожидание завершения
            history = await asyncio.get_event_loop().run_in_executor(
                None, poll_completion, prompt_id
            )
            if not history:
                raise RuntimeError("Таймаут генерации")

            # 4. Поиск выходного файла (приоритет — история, запасной — последний файл)
            output_file = find_output_from_history(history) or find_latest_output()
            if not output_file:
                raise RuntimeError("Выходной файл не найден")

            # 5. Пост-обработка
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Видео готово, улучшаю качество..."
            )
            final_path = await asyncio.get_event_loop().run_in_executor(
                None, postprocess_video, output_file
            )

            # 6. Отправка видео в Telegram
            file_size_mb = os.path.getsize(final_path) / (1024 * 1024)

            if file_size_mb > 50:
                await app.bot.send_message(
                    chat_id=job["chat_id"],
                    text=f"Видео слишком большое для Telegram ({file_size_mb:.0f} МБ).\n"
                         f"Сохранил на Google Drive!"
                )
            else:
                with open(final_path, "rb") as video_file:
                    await app.bot.send_video(
                        chat_id=job["chat_id"],
                        video=video_file,
                        caption="Готово!",
                        supports_streaming=True,
                    )

            # 7. Синхронизация на Drive
            saved = sync_file_to_drive(final_path)
            drive_msg = "\nСохранено на Google Drive!" if saved else ""

            keyboard = [[InlineKeyboardButton("Сделать ещё", callback_data="restart")]]
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text=f"Готово!{drive_msg}",
                reply_markup=InlineKeyboardMarkup(keyboard),
            )

            elapsed = (time.time() - start_time) / 60
            generation_times.append(elapsed)
            logger.info(f"Задача выполнена за {elapsed:.1f} мин")

        except Exception as e:
            logger.error(f"Задача не выполнена: {e}")
            keyboard = [[InlineKeyboardButton("Попробовать снова", callback_data="restart")]]
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Ой, что-то пошло не так.\nПопробуй ещё раз!",
                reply_markup=InlineKeyboardMarkup(keyboard),
            )

        finally:
            job_queue.task_done()

# ==================== ЗАПУСК ====================

async def post_init(app: Application) -> None:
    """Запуск воркера после инициализации бота."""
    asyncio.create_task(worker(app))
    logger.info("Бот запущен! Откройте бота в Telegram и нажмите /start")

print("Запускаю Telegram бота...")
print("Нажмите кнопку СТОП в Colab для остановки.\n")

app = Application.builder().token(BOT_TOKEN).post_init(post_init).build()

conv_handler = ConversationHandler(
    entry_points=[
        CommandHandler("start", start),
        CallbackQueryHandler(restart_entry, pattern="^restart$"),
    ],
    states={
        CHOOSE_MODE: [CallbackQueryHandler(choose_mode, pattern="^mode_")],
        WAIT_PHOTO: [MessageHandler(filters.PHOTO, receive_photo)],
        WAIT_PROMPT: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_prompt)],
        CHOOSE_DURATION: [CallbackQueryHandler(choose_duration, pattern="^dur_")],
        CHOOSE_ORIENTATION: [CallbackQueryHandler(choose_orientation, pattern="^orient_")],
    },
    fallbacks=[CommandHandler("cancel", cancel)],
    per_message=False,
)

app.add_handler(conv_handler)
app.add_handler(CommandHandler("queue", queue_status))

app.run_polling(allowed_updates=Update.ALL_TYPES)